In [1]:
# Load env variables

from dotenv import load_dotenv
load_dotenv()

# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)

    if not message.content:
        return ""
    return message.content[0].text


In [3]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01E3jTuyq6VHA5WxDx2Cxrkg', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='"', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='CustomerVault is a fictional relational database storing the personal profiles,', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' purchase histories, and 

In [4]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass

stream.get_final_message()


ParsedMessage(id='msg_01YPZjuTvAu3zGuXezrmQWuu', container=None, content=[ParsedTextBlock(citations=None, text='Here is a one sentence description of a fake database:\n\n**"NovaCorp Employee Registry"** is a fictional relational database containing fabricated records of 10,000 employees, including made-up names, departments, salaries, and office locations across imaginary branch offices worldwide.', type='text', parsed_output=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=64, server_tool_use=None, service_tier='standard'))

In [5]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json. Reply with ONLY a json code block.")
response = chat(messages, stop_sequences=["\n```"])
print(response)


```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["stopped"]
  }
}


In [6]:
import json

# response contains "```json\n{...}" — strip the opening fence before parsing
json_str = response.replace("```json", "").strip()
data = json.loads(json_str)
print(json.dumps(data, indent=2))


{
  "source": [
    "aws.ec2"
  ],
  "detail-type": [
    "EC2 Instance State-change Notification"
  ],
  "detail": {
    "state": [
      "stopped"
    ]
  }
}


In [27]:
messages = []

prompt = """Generate three different sample AWS CLI commands. Each should be a very short"""

add_user_message(messages, prompt)

text = chat(messages)
print(text.strip())

Here are three short AWS CLI commands:

1. **List S3 Buckets**
```bash
aws s3 ls
```

2. **Describe EC2 Instances**
```bash
aws ec2 describe-instances
```

3. **List Lambda Functions**
```bash
aws lambda list-functions
```


In [28]:
from IPython.display import Markdown

Markdown(text)

Here are three short AWS CLI commands:

1. **List S3 Buckets**
```bash
aws s3 ls
```

2. **Describe EC2 Instances**
```bash
aws ec2 describe-instances
```

3. **List Lambda Functions**
```bash
aws lambda list-functions
```